In [27]:
# ==========================================
# Student Health Risk Prediction
# CatBoost + Optuna (GPU)
# Kaggle Competition Notebook
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import json
import os
import sys
import random

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

import optuna
sys.path.append(os.path.abspath(".."))
from src.preprocessing import preprocess
from src.feature_engineering import create_features

In [22]:
# ==========================================
# Configuration
# ==========================================

RANDOM_STATE = 42

TARGET = "health_condition"

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [23]:
# ==========================================
# Load Dataset
# ==========================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)

train.head()

Train Shape : (690088, 15)
Test Shape  : (295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [24]:
# ==========================================
# Save IDs
# ==========================================

train_ids = train["id"].copy()

test_ids = test["id"].copy()

In [28]:
# ==========================================
# Preprocessing
# ==========================================

train = preprocess(train)
test = preprocess(test)

print(train.shape)
print(test.shape)

(690088, 14)
(295753, 13)


In [29]:
# ==========================================
# Feature Engineering V2
# ==========================================

train = create_features(train)
test = create_features(test)

print(train.shape)
print(test.shape)

(690088, 34)
(295753, 33)


In [13]:
%whos

Variable                  Type         Data/Info
------------------------------------------------
CatBoostClassifier        type         <class 'catboost.core.CatBoostClassifier'>
StratifiedKFold           ABCMeta      <class 'sklearn.model_sel<...>._split.StratifiedKFold'>
TARGET                    str          health_condition
TEST_PATH                 str          ../data/test.csv
TRAIN_PATH                str          ../data/train.csv
X                         DataFrame    Shape: (690088, 14)
balanced_accuracy_score   function     <function balanced_accura<...>_score at 0x76d47a05dd00>
cat_feature_indices       list         n=6
categorical_features      list         n=6
np                        module       <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
objective                 function     <function objective at 0x76d4e8135ee0>
optuna                    module       <module 'optuna' from '/h<...>ages/optuna/__init__.py'>
pd                        module       <module '

In [30]:
# ==========================================
# Prepare Features
# ==========================================

X = train.drop(columns=[TARGET]).copy()

y = train[TARGET].copy()

print("Features :", X.shape)

print("Target   :", y.shape)

Features : (690088, 33)
Target   : (690088,)


In [31]:
# ==========================================
# Detect Features
# ==========================================

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=[
        "int64",
        "float64",
        "int32",
        "float32"
    ]
).columns.tolist()

print("Categorical Features")

print(categorical_features)

print()

print("Numerical Features")

print(numerical_features)

Categorical Features
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category', 'bmi_risk', 'stress_sleep', 'activity_diet', 'gender_activity', 'smoking_stress', 'diet_smoking']

Numerical Features
['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'activity_score', 'calories_per_step', 'water_per_exercise', 'exercise_sleep_ratio', 'steps_per_minute', 'calories_per_minute', 'activity_density', 'heart_activity', 'heart_steps', 'bmi_sleep', 'bmi_water', 'hydration_score', 'health_index']


In [32]:
# ==========================================
# Clean Missing Values
# ==========================================

for col in categorical_features:

    X[col] = (
        X[col]
        .fillna("Missing")
        .astype(str)
    )

for col in numerical_features:

    X[col] = X[col].fillna(
        X[col].median()
    )

print("Categorical Missing Values")

print(
    X[categorical_features]
    .isna()
    .sum()
    .sum()
)

print()

print("Numerical Missing Values")

print(
    X[numerical_features]
    .isna()
    .sum()
    .sum()
)

Categorical Missing Values
0

Numerical Missing Values
0


In [33]:
# ==========================================
# Final Check
# ==========================================

print("="*60)

print("Train Shape")

print(X.shape)

print()

print("Target Shape")

print(y.shape)

print()

print("Number of Features")

print(X.shape[1])

print()

print("Categorical Features")

print(len(categorical_features))

print()

print("Numerical Features")

print(len(numerical_features))

print("="*60)

Train Shape
(690088, 33)

Target Shape
(690088,)

Number of Features
33

Categorical Features
13

Numerical Features
20


In [34]:
# ==========================================
# Optuna Objective Function
# ==========================================

def objective(trial):

    bootstrap_type = trial.suggest_categorical(
        "bootstrap_type",
        ["Bayesian", "Bernoulli"]
    )

    params = {

        # GPU
        "task_type": "GPU",
        "devices": "0",

        # Objective
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",

        # Fixed
        "iterations": 1200,
        "random_seed": RANDOM_STATE,
        "verbose": False,
        "auto_class_weights": "Balanced",

        # Search Space
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.10,
            log=True
        ),

        "depth": trial.suggest_int(
            "depth",
            5,
            10
        ),

        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg",
            1,
            15
        ),

        "random_strength": trial.suggest_float(
            "random_strength",
            0,
            5
        ),

        "border_count": trial.suggest_int(
            "border_count",
            64,
            255
        ),

        "grow_policy": trial.suggest_categorical(
            "grow_policy",
            ["SymmetricTree", "Depthwise"]
        ),

        "leaf_estimation_iterations": trial.suggest_int(
            "leaf_estimation_iterations",
            1,
            10
        ),

        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf",
            1,
            32
        ),

        "bootstrap_type": bootstrap_type
    }

    if bootstrap_type == "Bayesian":

        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature",
            0,
            10
        )

    else:

        params["subsample"] = trial.suggest_float(
            "subsample",
            0.6,
            1.0
        )

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            cat_features=categorical_features,
            use_best_model=True,
            early_stopping_rounds=150
        )

        pred = model.predict(X_valid)

        score = balanced_accuracy_score(
            y_valid,
            pred
        )

        scores.append(score)

    return np.mean(scores)

In [35]:
# ==========================================
# Create Optuna Study
# ==========================================

study = optuna.create_study(

    direction="maximize",

    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
        multivariate=True
    ),

    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10
    )
)

print("Study Created Successfully")

[I 2026-07-29 19:42:17,761] A new study created in memory with name: no-name-28ca6f33-2ef6-4cdb-b9b2-09029cb20704


Study Created Successfully


In [36]:
study.optimize(
    objective,
    n_trials=1,
    show_progress_bar=True
)

  0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-07-29 19:43:53,812] Trial 0 finished with value: 0.9484940395215166 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.05395030966670229, 'depth': 8, 'l2_leaf_reg': 3.1842609661941115, 'random_strength': 0.7799726016810132, 'border_count': 75, 'grow_policy': 'SymmetricTree', 'leaf_estimation_iterations': 8, 'min_data_in_leaf': 1, 'subsample': 0.9879639408647978}. Best is trial 0 with value: 0.9484940395215166.


In [37]:
print(study.best_value)
print(study.best_params)

0.9484940395215166
{'bootstrap_type': 'Bernoulli', 'learning_rate': 0.05395030966670229, 'depth': 8, 'l2_leaf_reg': 3.1842609661941115, 'random_strength': 0.7799726016810132, 'border_count': 75, 'grow_policy': 'SymmetricTree', 'leaf_estimation_iterations': 8, 'min_data_in_leaf': 1, 'subsample': 0.9879639408647978}


In [38]:
# ==========================================
# Run Optuna Optimization
# ==========================================

N_TRIALS = 75

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

  0%|          | 0/75 [00:00<?, ?it/s]

[I 2026-07-29 19:47:08,475] Trial 1 finished with value: 0.9481123776788093 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.015199348301309814, 'depth': 6, 'l2_leaf_reg': 5.259391401433528, 'random_strength': 2.6237821581611893, 'border_count': 146, 'grow_policy': 'Depthwise', 'leaf_estimation_iterations': 2, 'min_data_in_leaf': 10, 'bagging_temperature': 3.663618432936917}. Best is trial 0 with value: 0.9484940395215166.
[I 2026-07-29 19:51:02,011] Trial 2 finished with value: 0.948913816935996 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.015837031559118753, 'depth': 8, 'l2_leaf_reg': 9.293803964068594, 'random_strength': 0.23225206359998862, 'border_count': 180, 'grow_policy': 'SymmetricTree', 'leaf_estimation_iterations': 10, 'min_data_in_leaf': 31, 'subsample': 0.9233589392465844}. Best is trial 2 with value: 0.948913816935996.
[I 2026-07-29 19:53:57,924] Trial 3 finished with value: 0.9479959339142917 and parameters: {'bootstrap_type': 'Baye

In [44]:
print(study.best_value)
print(study.best_params)

0.9493068522610774
{'bootstrap_type': 'Bernoulli', 'learning_rate': 0.03309416821002669, 'depth': 7, 'l2_leaf_reg': 10.801507310686478, 'random_strength': 0.898531201248234, 'border_count': 253, 'grow_policy': 'Depthwise', 'leaf_estimation_iterations': 9, 'min_data_in_leaf': 25, 'subsample': 0.6828948177465322}


In [39]:
# ==========================================
# Best Trial
# ==========================================

print("=" * 60)

print("Best CV Score")
print(study.best_value)

print()

print("Best Parameters")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

print("=" * 60)

Best CV Score
0.9493068522610774

Best Parameters
bootstrap_type: Bernoulli
learning_rate: 0.03309416821002669
depth: 7
l2_leaf_reg: 10.801507310686478
random_strength: 0.898531201248234
border_count: 253
grow_policy: Depthwise
leaf_estimation_iterations: 9
min_data_in_leaf: 25
subsample: 0.6828948177465322


In [45]:
# ==========================================
# Save Best Parameters
# ==========================================

best_params = study.best_params.copy()

best_params.update({

    "iterations": 1200,
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",

    "task_type": "GPU",
    "devices": "0",

    "random_seed": RANDOM_STATE,
    "verbose": 200,

    "auto_class_weights": "Balanced"

})

with open("../output/models/best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Saved: models/best_params.json")

Saved: models/best_params.json


In [41]:
# Optimization History
import optuna.visualization as vis

vis.plot_optimization_history(study)

In [42]:
# Parameter Importance
vis.plot_param_importances(study)

In [43]:
# Parallel Coordinate Plot
vis.plot_parallel_coordinate(study)

[W 2026-07-29 22:30:37,926] Your study has only completed trials with missing parameters.


In [46]:
# ==========================================
# Best Parameters from Optuna
# ==========================================

best_params = {
    "bootstrap_type": "Bernoulli",
    "learning_rate": 0.03309416821002669,
    "depth": 7,
    "l2_leaf_reg": 10.801507310686478,
    "random_strength": 0.898531201248234,
    "border_count": 253,
    "grow_policy": "Depthwise",
    "leaf_estimation_iterations": 9,
    "min_data_in_leaf": 25,
    "subsample": 0.6828948177465322,

    # Fixed Parameters
    "iterations": 1200,
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",

    "task_type": "GPU",
    "devices": "0",

    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "verbose": False
}

In [47]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(**best_params)

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=150
    )

    pred = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, pred)

    scores.append(score)

    print(f"Fold {fold}: {score:.6f}")

print("\nMean CV:", np.mean(scores))
print("Std CV :", np.std(scores))

Fold 1: 0.949997
Fold 2: 0.951262
Fold 3: 0.948970
Fold 4: 0.949322
Fold 5: 0.947685

Mean CV: 0.9494470167050434
Std CV : 0.0011783808015736609


In [48]:
# Train Final Model
final_model = CatBoostClassifier(**best_params)

final_model.fit(
    X,
    y,
    cat_features=categorical_features
)

CatBoostClassifier(auto_class_weights='Balanced', bootstrap_type='Bernoulli', border_count=253, depth=7, devices='0', eval_metric='MultiClass', grow_policy='Depthwise', iterations=1200, l2_leaf_reg=10.801507310686478, leaf_estimation_iterations=9, learning_rate=0.03309416821002669, loss_function='MultiClass', min_data_in_leaf=25, random_seed=42, random_strength=0.898531201248234, subsample=0.6828948177465322, task_type='GPU', verbose=False)

In [64]:
# Save Model

os.makedirs("models", exist_ok=True)

final_model.save_model("../output/models/shrp_catboost_pro_v1_0.94985.cbm")

print("Model save.")

Model save.


In [52]:
feature_importance = pd.DataFrame({

    "Feature": X.columns,
    "Importance": final_model.get_feature_importance()

})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(50)

,Feature,Importance
0,sleep_duration,42.909324
8,stress_level,22.579465
2,bmi,15.338961
10,physical_activity_level,7.986294
28,stress_sleep,1.560838
30,gender_activity,1.308785
31,smoking_stress,1.006550
22,bmi_sleep,0.782306
11,smoking_alcohol,0.562747
6,water_intake,0.525334


In [53]:
# save the Feature 
feature_importance.to_csv(
    "../output/models/feature_importance.csv",
    index=False
)

In [57]:
print(test.columns)
print(test_processed.columns)

Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake', 'diet_type',
       'stress_level', 'sleep_quality', 'physical_activity_level',
       'smoking_alcohol', 'gender', 'activity_score', 'calories_per_step',
       'water_per_exercise', 'exercise_sleep_ratio', 'steps_per_minute',
       'calories_per_minute', 'activity_density', 'heart_activity',
       'heart_steps', 'bmi_sleep', 'bmi_water', 'hydration_score',
       'health_index', 'bmi_category', 'bmi_risk', 'stress_sleep',
       'activity_diet', 'gender_activity', 'smoking_stress', 'diet_smoking'],
      dtype='str')
Index(['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake', 'diet_type',
       'stress_level', 'sleep_quality', 'physical_activity_level',
       'smoking_alcohol', 'gender', 'activity_score', 'calories_per_step',
       'water_per_exercise', 'exercise_sleep_ratio', 'steps_per_

In [58]:
print(test.head())
print(test.shape)

   sleep_duration  heart_rate    bmi  calorie_expenditure  step_count  \
0            5.35        64.9  23.48               2745.0     14167.0   
1            6.99        83.1  22.42               1773.0      6801.0   
2            6.68        59.7  24.14               3040.0     13250.0   
3            7.13        78.5  26.26               2494.0      6331.0   
4            5.49        77.7  23.29               1828.0     13894.0   

   exercise_duration  water_intake diet_type stress_level sleep_quality  ...  \
0               59.5          1.86       veg         high          poor  ...   
1               24.5          2.40  balanced         high          poor  ...   
2               48.5          2.76  balanced       medium          poor  ...   
3               56.9          2.34       veg          low          good  ...   
4               39.4          2.45       veg         high       average  ...   

  bmi_water hydration_score health_index  bmi_category  bmi_risk  \
0   43.6728 

In [60]:
# ==========================================
# Preprocess Test Data
# ==========================================

# test_ids was already saved earlier:
# test_ids = test["id"].copy()

test_processed = preprocess(test.copy())
test_processed = create_features(test_processed)

X_test = test_processed.copy()

# Drop id only if it still exists
if "id" in X_test.columns:
    X_test = X_test.drop(columns=["id"])

# Ensure same feature order as training
X_test = X_test[X.columns]

In [65]:
predictions = final_model.predict(X_test).ravel()

submission = pd.DataFrame({
    "id": test_ids,
    "health_condition": predictions
})

submission.to_csv("../output/submissions/submission_shrp_catboost_pro_0.94985.csv", index=False)

submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
